In [1]:
# ==========================================
# Notebook 05
# Vector Index Construction
# ==========================================

import pandas as pd
import numpy as np

import faiss

from sklearn.metrics.pairwise import cosine_similarity

In [2]:
profiles_df = pd.read_csv("../data/item_profiles.csv")

profiles_df.head()

,item_id,title,category,tags,content_profile
0,1,The Early Days of Stripe,Startup,startup fintech founders entrepreneurship vent...,Title: The Early Days of Stripe Category: Star...
1,2,Building SpaceX,Startup,startup fintech founders entrepreneurship vent...,Title: Building SpaceX Category: Startup Tags:...
2,3,AI for Healthcare,Artificial Intelligence,ai machine-learning llm deep-learning transfor...,Title: AI for Healthcare Category: Artificial ...
3,4,The Psychology of Habits,Wellness,habits productivity self-improvement,Title: The Psychology of Habits Category: Well...
4,5,Cloud Computing Fundamentals,Technology,cloud computing distributed-systems scalability,Title: Cloud Computing Fundamentals Category: ...


In [3]:
item_embeddings = np.load("../data/item_embeddings.npy")

item_embeddings.shape

(10, 384)

In [4]:
print("Total Items:", len(profiles_df))

print()

print("Embedding Shape:", item_embeddings.shape)

Total Items: 10

Embedding Shape: (10, 384)


In [6]:
item_embeddings = item_embeddings.astype("float32")

In [7]:
item_embeddings.dtype

dtype('float32')

In [9]:
embedding_dimension = item_embeddings.shape[1]

embedding_dimension

384

In [10]:
index = faiss.IndexFlatL2(embedding_dimension)

In [11]:
index.ntotal

0

In [12]:
query_embedding = item_embeddings[0].reshape(1, -1)

In [13]:
distances, indices = index.search(query_embedding, 5)

In [14]:
distances

array([[3.4028235e+38, 3.4028235e+38, 3.4028235e+38, 3.4028235e+38,
        3.4028235e+38]], dtype=float32)

In [15]:
indices

array([[-1, -1, -1, -1, -1]], dtype=int64)

In [16]:
for idx in indices[0]:

    print()

    print(profiles_df.iloc[idx]["title"])


Large Language Models

Large Language Models

Large Language Models

Large Language Models

Large Language Models


In [17]:
def search_similar_items(query_embedding, top_k=5):

    distances, indices = index.search(query_embedding.reshape(1, -1), top_k)

    results = []

    for rank, idx in enumerate(indices[0]):

        results.append(
            {
                "rank": rank + 1,
                "item_id": profiles_df.iloc[idx]["item_id"],
                "title": profiles_df.iloc[idx]["title"],
                "category": profiles_df.iloc[idx]["category"],
                "distance": float(distances[0][rank]),
            }
        )

    return pd.DataFrame(results)

In [18]:
search_similar_items(item_embeddings[0], top_k=5)

,rank,item_id,title,category,distance
0,1,10,Large Language Models,Artificial Intelligence,3.402823e+38
1,2,10,Large Language Models,Artificial Intelligence,3.402823e+38
2,3,10,Large Language Models,Artificial Intelligence,3.402823e+38
3,4,10,Large Language Models,Artificial Intelligence,3.402823e+38
4,5,10,Large Language Models,Artificial Intelligence,3.402823e+38


In [19]:
search_similar_items(item_embeddings[2], top_k=5)

,rank,item_id,title,category,distance
0,1,10,Large Language Models,Artificial Intelligence,3.402823e+38
1,2,10,Large Language Models,Artificial Intelligence,3.402823e+38
2,3,10,Large Language Models,Artificial Intelligence,3.402823e+38
3,4,10,Large Language Models,Artificial Intelligence,3.402823e+38
4,5,10,Large Language Models,Artificial Intelligence,3.402823e+38


In [20]:
target_item = profiles_df[
    profiles_df["title"].str.contains("Large Language Models", case=False)
].iloc[0]

target_item

item_id                                                           10
title                                          Large Language Models
category                                     Artificial Intelligence
tags               ai machine-learning llm deep-learning transfor...
content_profile    Title: Large Language Models Category: Artific...
Name: 9, dtype: object

In [21]:
target_index = target_item.name

target_index

9

In [22]:
search_similar_items(item_embeddings[target_index], top_k=5)

,rank,item_id,title,category,distance
0,1,10,Large Language Models,Artificial Intelligence,3.402823e+38
1,2,10,Large Language Models,Artificial Intelligence,3.402823e+38
2,3,10,Large Language Models,Artificial Intelligence,3.402823e+38
3,4,10,Large Language Models,Artificial Intelligence,3.402823e+38
4,5,10,Large Language Models,Artificial Intelligence,3.402823e+38


In [23]:
similarities = cosine_similarity(
    item_embeddings[target_index].reshape(1, -1), item_embeddings
)[0]

In [24]:
ranked_idx = np.argsort(similarities)[::-1][:5]

ranked_idx

array([9, 2, 6, 5, 4], dtype=int64)

In [25]:
for idx in ranked_idx:

    print()

    print(profiles_df.iloc[idx]["title"])

    print(round(similarities[idx], 4))


Large Language Models
1.0

AI for Healthcare
0.6894

Machine Learning in Finance
0.4624

Modern Data Engineering
0.3567

Cloud Computing Fundamentals
0.2538


In [26]:
faiss.write_index(index, "../data/item_index.faiss")

In [27]:
loaded_index = faiss.read_index("../data/item_index.faiss")

In [28]:
loaded_index.ntotal

0

In [29]:
distances, indices = loaded_index.search(query_embedding, 5)

indices

array([[-1, -1, -1, -1, -1]], dtype=int64)

In [30]:
index_metadata = profiles_df[["item_id", "title", "category"]]

In [31]:
index_metadata.head()

,item_id,title,category
0,1,The Early Days of Stripe,Startup
1,2,Building SpaceX,Startup
2,3,AI for Healthcare,Artificial Intelligence
3,4,The Psychology of Habits,Wellness
4,5,Cloud Computing Fundamentals,Technology


In [32]:
index_metadata.to_csv("../data/item_index_metadata.csv", index=False)

In [33]:
print("Index Saved Successfully")

Index Saved Successfully
